## 1. Importing Libraries and Datasets

In [3]:
#importing libraries

import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os

In [4]:
#importing dataset from kaggle and selecting relevant columns

path = r"C:\Users\ishik\.cache\kagglehub\datasets\rounakbanik\the-movies-dataset\versions\7"
movies_df = pd.read_csv(
    path + "/movies_metadata.csv",
    low_memory=False
)
movies_df = movies_df[
    ["title", "overview", "genres"]
].copy()
movies_df.head()

,title,overview,genres
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ..."
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]"


## 2. Creating Word Embeddings

### 2.1 Cleaning

In [7]:
movies_df = movies_df.dropna(subset=["overview"]).reset_index(drop=True)
movies_df.shape

(44512, 3)

In [8]:
def clean_text(text):
    text=text.lower()
    text=re.sub(r"[^A-Za-z\s]","",text)
    tokens=text.split()
    return tokens

In [9]:
corpus = movies_df["overview"].apply(clean_text)
corpus.head()

0    [led, by, woody, andys, toys, live, happily, i...
1    [when, siblings, judy, and, peter, discover, a...
2    [a, family, wedding, reignites, the, ancient, ...
3    [cheated, on, mistreated, and, stepped, on, th...
4    [just, when, george, banks, has, recovered, fr...
Name: overview, dtype: object

### 2.2: Generating Word Pairs for Embedding Training

In [10]:
from collections import Counter
word_counts = Counter()
for sentence in corpus:
    word_counts.update(sentence)

len(word_counts)

88932

In [11]:
#Remove rare words
vocab_words = [
    word
    for word, count in word_counts.items()
    if count >= 5
]

len(vocab_words)

23206

In [12]:
word_to_id = {
    word: idx
    for idx, word in enumerate(vocab_words)
}

id_to_word = {
    idx: word
    for word, idx in word_to_id.items()
}

In [13]:
list(word_to_id.items())[:3]

[('led', 0), ('by', 1), ('woody', 2)]

In [14]:
list(id_to_word.items())[:3]

[(0, 'led'), (1, 'by'), (2, 'woody')]

In [18]:
corpus_ids = []

for sentence in corpus:

    sentence_ids = [
        word_to_id[word]
        for word in sentence
        if word in word_to_id
    ]

    corpus_ids.append(sentence_ids)

In [24]:
len(corpus_ids) #list of lists, with list of id's for everyy overview (sentence)

44512

In [25]:
def generate_pairs(sentence_ids):

    pairs = []

    for i in range(len(sentence_ids)):

        center = sentence_ids[i]

        if i > 0:
            pairs.append(
                (center, sentence_ids[i-1])
            )

        if i < len(sentence_ids)-1:
            pairs.append(
                (center, sentence_ids[i+1])
            )

    return pairs

In [26]:
all_pairs = []

for sentence_ids in corpus_ids:

    pairs = generate_pairs(sentence_ids)

    all_pairs.extend(pairs)

In [27]:
len(all_pairs)

4579326

In [28]:
all_pairs[:5]

[(0, 1), (1, 0), (1, 2), (2, 1), (2, 3)]

In [30]:
# Converting pairs to pytorch tensors
inputs = [pair[0] for pair in all_pairs]
targets = [pair[1] for pair in all_pairs]

In [31]:
#Convert to tensors
import torch

X = torch.tensor(inputs)
y = torch.tensor(targets)

In [36]:
print("Number of Pairs: ", y.shape)
print("vocab_size: ", len(word_to_id))

Number of Pairs:  torch.Size([4579326])
vocab_size:  23206
